# Session 2 — cleaning & transforming (Titanic)

Real data is never perfect: values are missing, columns are duplicated, names are unclear.
In this session you clean the Titanic data and then answer questions with `groupby`.

**You will learn to:**
- find and handle missing values
- keep, rename and convert columns
- create new columns, including buckets with `pd.cut`
- summarise data by group with `groupby` and `agg`

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

## Part 1 — cleaning

The first question with any dataset: **what is missing?**
`isna()` marks every missing cell as `True`; `.sum()` counts the `True` values per column.
`.mean()` instead gives the share that is missing.

In [ ]:
df.isna().sum()

In [ ]:
(df.isna().mean() * 100).round(1)   # % missing per column

There are three ways to handle missing data — choose per column:

| Situation | Action | Method |
|---|---|---|
| Most values missing | drop the **column** | `drop(columns=[...])` |
| Very few rows missing | drop those **rows** | `dropna(subset=[...])` |
| Some missing, column is important | **fill** with a sensible value | `fillna(...)` |

In [ ]:
# deck: ~77% missing -> useless, drop the column
df = df.drop(columns=["deck"])

In [ ]:
# embark_town: only 2 missing -> drop those rows
df = df.dropna(subset=["embark_town"])
df.shape

In [ ]:
# age: ~20% missing, but important -> fill with the median
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
df["age"].isna().sum()

**Important:** pandas methods return a *new* result — they don't change `df` by themselves.
That's why we always **reassign**: `df = df.drop(...)`, `df["age"] = df["age"].fillna(...)`.

### Keep only useful columns, rename, change types

This dataset repeats information (`class` = `pclass`, `alive` = `survived`, `embarked` = `embark_town`).
Selecting a list of columns keeps only those. `rename` gives columns clearer names. `astype` changes a column's data type.

In [ ]:
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embark_town"]]
df.head()

In [ ]:
df = df.rename(columns={"sibsp": "siblings_spouses", "parch": "parents_children"})
df.columns

In [ ]:
df["age"] = df["age"].astype(int)   # 0.42 years -> 0; fine for our purpose
df.dtypes

## Part 2 — new columns & groupby

A new column is just a calculation on whole columns at once — no loops needed.

In [ ]:
df["family_size"] = df["siblings_spouses"] + df["parents_children"] + 1
df["fare_per_person"] = (df["fare"] / df["family_size"]).round(2)
df.head()

`apply` runs your own function on every value. It's flexible, but slower —
use plain column calculations (like above) whenever you can.

In [ ]:
df["is_child"] = df["age"].apply(lambda a: a < 18)   # same result as: df["age"] < 18 (the faster way)
df["is_child"].value_counts()

`pd.cut` turns a number into named buckets (ranges). Very common before making charts.
Each bucket includes its right edge: with the bins below, age 12 is still a `child` and age 18 is still a `teen`.
`include_lowest=True` puts the very first edge (age 0) into the first bucket too.

In [ ]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["child", "teen", "young adult", "adult", "senior"],
    include_lowest=True,
)
df["age_group"].value_counts()

### groupby = split → apply → combine

`df.groupby("sex")["survived"].mean()` does three things:
1. **split** the table into one group per sex
2. **apply** `mean()` to `survived` inside each group
3. **combine** the results into a small table

It's the same idea as a pivot table in Excel.

In [ ]:
df.groupby("sex")["survived"].mean()

You can group by several columns. `unstack()` turns the inner group into columns, giving a readable table.

In [ ]:
df.groupby(["pclass", "sex"])["survived"].mean()

In [ ]:
df.groupby(["pclass", "sex"])["survived"].mean().unstack().round(2)

`agg` computes several summaries at once. Each one is written as `new_name=(column, function)`.

In [ ]:
summary = df.groupby("pclass").agg(
    passengers=("survived", "size"),
    survival_rate=("survived", "mean"),
    avg_age=("age", "mean"),
    avg_fare=("fare", "mean"),
).round(2)
summary

When grouping by a bucket column made with `pd.cut`, add `observed=True` so only groups that actually exist are shown.

In [ ]:
df.groupby("age_group", observed=True)["survived"].mean().round(2)

## Exercises
Start again from the raw data: `raw = sns.load_dataset("titanic")`

1. Count missing values per column.
2. Fill missing `age` with the median and drop rows without `embark_town`. Check that both columns have 0 missing.
3. Add a `family_size` column. What is the survival rate for each family size?
4. Show survival rate by `embark_town` (rows) and `sex` (columns).

*Try each one yourself first, then compare with `solutions.ipynb` in this folder.*

## Summary
- `isna().sum()` → decide: drop column / drop rows / fill
- always reassign: `df = df.something()`
- new column = column math; `pd.cut` for buckets
- `groupby(...)[col].mean()`, `.unstack()`, `.agg(...)`

## Practice
1. Add `fare_group` with `pd.qcut(df["fare"], 4, labels=["low", "mid", "high", "very high"])` (4 equal-sized groups). Survival rate per fare group?
2. Average age of survivors vs. non-survivors, per class (tip: `groupby(["pclass", "survived"])`).
3. Save your clean DataFrame with `df.to_csv("titanic_clean.csv", index=False)` and find the file (in Colab: the Files panel on the left).

*Try each one yourself first, then compare with `solutions.ipynb` in this folder.*

## LeetCode practice

On LeetCode choose **Pandas** as the language. You get a function that receives DataFrames — write your code inside it and `return` the answer as a DataFrame.

2887. [Fill Missing Data](https://leetcode.com/problems/fill-missing-data/). `fillna`.
1873. [Calculate Special Bonus](https://leetcode.com/problems/calculate-special-bonus/). A new column from a condition; `str.startswith` checks the first letter.
1393. [Capital Gain/Loss](https://leetcode.com/problems/capital-gainloss/). Make buys negative in a new column, then `groupby` + `sum`.

LeetCode checks your answer, and each problem has an *Editorial* / *Solutions* tab if you get stuck.